# Does a small model know what it doesn't know?

Calibration study of an open language model, 4-bit, on a single RTX 4070 (12 GB).

**This notebook currently covers Stage 1, step a1 (protocol §0-§1):** load each model in 4-bit and run one
forward pass. Confirm it fits in VRAM and that the logits tensor prints. Nothing else matters until this works.

Later steps (a2 dataset, a3 scoring function, a4 `scores.csv`, ...) are added below as they are built.

## §0 Environment check

The GPU must be visible to PyTorch, and `bitsandbytes` must be built against the right CUDA.

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows without Developer Mode: harmless cache warning

import torch

print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
assert torch.cuda.is_available(), "No CUDA device visible - install the cu124 build of torch (protocol §0)"

props = torch.cuda.get_device_properties(0)
print("GPU            :", props.name)
print("VRAM total     :", round(props.total_memory / 1e9, 1), "GB")

import transformers, bitsandbytes
print("transformers   :", transformers.__version__)
print("bitsandbytes   :", bitsandbytes.__version__)

torch          : 2.6.0+cu124
CUDA available : True
GPU            : NVIDIA GeForce RTX 4070
VRAM total     : 12.9 GB


C:\Users\billy\Documents\Artificial_Confidence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers   : 5.17.0
bitsandbytes   : 0.50.2


## §1 The two models

| Variant | Hugging Face repo | Notes |
|---|---|---|
| `stock` | [Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) | Official model. BF16 safetensors, about 16 GB. |
| `uncensored` | [huihui-ai/Huihui-Qwen3-8B-abliterated-v2](https://huggingface.co/huihui-ai/Huihui-Qwen3-8B-abliterated-v2) | Abliterated Qwen3-8B (refusal behaviour removed by editing weights, not by training). Same base, architecture and parameter count. BF16 safetensors, about 16 GB. |

Both are downloaded from the Hub on first use into `C:\Users\<you>\.cache\huggingface\hub` (about 32 GB
total) and quantised on the fly to nf4 with the **identical** `BitsAndBytesConfig`. GGUF files are **not** used:
the protocol needs raw logits from `transformers`.

**The two models are never in memory at the same time.** Each variant is loaded, scored, then freed before the
next one loads. Two reasons:

1. **It does not fit.** Each is roughly 5-6 GB at nf4, so two together would be about 11-12 GB on a 12 GB card that
   is also driving your display, with nothing left for activations.
2. **It is not needed.** Each variant's scores are computed independently and stored as rows in `scores.csv`
   (the `model` column). The stock-versus-uncensored comparison (struggle S4) is made afterwards, from the CSV.

Everything except `MODELS[variant]` is identical between the two runs, so any difference between conditions
is not caused by loading.

In [2]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELS = {
    "stock":      "Qwen/Qwen3-8B",
    "uncensored": "huihui-ai/Huihui-Qwen3-8B-abliterated-v2",
}
VARIANTS = ["stock", "uncensored"]      # run one or both; they always run one at a time
LETTERS = ["A", "B", "C", "D"]

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# One hard-coded question, only to confirm the model runs (the real dataset is step a2).
QUESTION = "Which of these is a source of light?"
OPTIONS  = ["The Moon", "A mirror", "The Sun", "A window"]


def build_prompt(tok, question, options):
    body = "\n".join(f"{L}. {t}" for L, t in zip(LETTERS, options))
    user = ("Answer the multiple-choice question with a single letter.\n\n"
            f"Question: {question}\n{body}")
    return tok.apply_chat_template(
        [{"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=False,           # Qwen3: suppress the <think> block
    ) + "Answer:"

## Step a1: load, one forward pass, free

`run_a1(variant)` loads one model, runs a single forward pass, prints the logits, and then **frees the GPU** before
returning. It hands back only plain Python values (no tensors), so nothing keeps the model alive.
`enable_thinking=False` stops Qwen3 opening a `<think>` block; if your installed `transformers` does not support it,
this is where it will show up.

In [3]:
def run_a1(variant):
    model_id = MODELS[variant]
    print("=" * 78)
    print(f"[{variant}]  {model_id}")
    print("=" * 78)

    torch.cuda.reset_peak_memory_stats()
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map="cuda:0",
    )
    model.eval()
    footprint = model.get_memory_footprint() / 1e9
    print(f"model footprint : {footprint:.2f} GB   (expect roughly 5-6 GB for an 8B model at nf4)")

    letter_ids = [tok.encode(" " + L, add_special_tokens=False)[-1] for L in LETTERS]
    prompt = build_prompt(tok, QUESTION, OPTIONS)
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits          # shape: [batch, sequence, vocab]

    last = logits[0, -1].float()
    print("logits tensor shape :", tuple(logits.shape))
    print("last-position logits:", last)

    top = torch.topk(torch.softmax(last, dim=-1), 5)
    top5 = [(tok.convert_ids_to_tokens(i), i, round(p, 4)) for p, i in zip(top.values.tolist(), top.indices.tolist())]
    print("top-5 next tokens (token, id, prob):")
    for t, i, p in top5:
        print(f"   {p:6.3f}  id={i:<7} {t!r}")

    peak = torch.cuda.max_memory_allocated() / 1e9
    result = dict(
        variant=variant, model_id=model_id, footprint_gb=round(footprint, 2), peak_vram_gb=round(peak, 2),
        logits_shape=tuple(logits.shape), letter_ids=letter_ids,
        letter_tokens=tok.convert_ids_to_tokens(letter_ids), prompt=prompt, top5=top5,
    )

    # free the GPU so the next variant loads into an empty card
    del model, inputs, logits, last, top
    gc.collect()
    torch.cuda.empty_cache()
    result["vram_after_free_gb"] = round(torch.cuda.memory_allocated() / 1e9, 2)
    print(f"peak VRAM       : {peak:.2f} GB of {props.total_memory / 1e9:.1f} GB")
    print(f"after freeing   : {result['vram_after_free_gb']:.2f} GB still allocated")
    return result


results = {}
for v in VARIANTS:                 # strictly sequential: one model in memory at a time
    results[v] = run_a1(v)

[stock]  Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/399 [00:01<08:35,  1.30s/it]

Loading weights:   1%|          | 2/399 [00:02<08:05,  1.22s/it]

Loading weights:   1%|          | 4/399 [00:02<03:28,  1.89it/s]

...

Loading weights:  97%|█████████▋| 389/399 [00:16<00:00, 26.37it/s]

Loading weights:  98%|█████████▊| 392/399 [00:16<00:00, 22.38it/s]

Loading weights: 100%|██████████| 399/399 [00:16<00:00, 23.65it/s]

model footprint : 5.96 GB   (expect roughly 5-6 GB for an 8B model at nf4)


logits tensor shape : (1, 54, 151936)
last-position logits: tensor([ -2.5625, -15.1250, -13.8125,  ...,  -5.4062,  -5.4062,  -5.4062],
       device='cuda:0')
top-5 next tokens (token, id, prob):
    1.000  id=3070    'Ġ**'
    0.000  id=356     'ĠC'
    0.000  id=57960   'Ġ$\\'
    0.000  id=1124    'Ġ\\'
    0.000  id=1304    'Ġ__'
peak VRAM       : 6.18 GB of 12.9 GB
after freeing   : 0.01 GB still allocated


## a1 summary and checks

In [4]:
import pandas as pd

summary = pd.DataFrame(
    [{k: r[k] for k in ("variant", "model_id", "footprint_gb", "peak_vram_gb", "vram_after_free_gb")} for r in results.values()]
).set_index("variant")
display(summary)

total = props.total_memory / 1e9
for r in results.values():
    assert r["peak_vram_gb"] < total * 0.95, f"{r['variant']}: too close to the VRAM limit - consider a 4B model (protocol §1)"
    if r["vram_after_free_gb"] > 1.0:
        print(f"WARNING [{r['variant']}]: {r['vram_after_free_gb']} GB still allocated after freeing - "
              "the next model may not fit. Restart the kernel and run one variant at a time.")

if len(results) == 2:
    a, b = results["stock"], results["uncensored"]
    print("letter token ids identical :", a["letter_ids"] == b["letter_ids"], a["letter_tokens"])
    print("chat-template prompt identical:", a["prompt"] == b["prompt"])
    assert a["letter_ids"] == b["letter_ids"] and a["prompt"] == b["prompt"], \
        "The two variants tokenise differently - the comparison would be confounded"

print("a1 done: each model loads in 4-bit, a forward pass runs, logits print, and the GPU is freed between models.")

,model_id,footprint_gb,peak_vram_gb,vram_after_free_gb
variant,,,,
stock,Qwen/Qwen3-8B,5.96,6.17,0.01
uncensored,huihui-ai/Huihui-Qwen3-8B-abliterated-v2,5.96,6.18,0.01


letter token ids identical : True ['ĠA', 'ĠB', 'ĠC', 'ĠD']
chat-template prompt identical: True
a1 done: each model loads in 4-bit, a forward pass runs, logits print, and the GPU is freed between models.


## What to check before moving on to a2

- Footprint is roughly 5-6 GB per model and peak VRAM is comfortably under 12 GB.
- The top next-token candidates look like an answer (a letter with a leading space, e.g. `'ĠC'`), not `<think>`
  or garbage. If you see `<think>`, upgrade `transformers` or append the template's empty-think marker manually.
- The "after freeing" number is close to 0 GB, so the second model really loaded into an empty card.
- The two variants agree on letter token ids and prompt text (asserted above).
- Note anything that broke (version drift in `transformers` / `bitsandbytes` is common) in `notes/breakage-log.md`.

## §2 The dataset (step a2)

Load 500-1000 four-option questions. The benchmark is **ARC-Easy** (`allenai/ai2_arc`, config `ARC-Easy`,
`test` split) — science questions, multiple choice.

**Why the filter matters:** ARC mixes in items with 3 or 5 choices, and a few label their options `1-4`
instead of `A-D`. An unfiltered row still produces a confidence number — a meaningless one — and nothing
errors, so it would silently corrupt the results. `usable()` keeps only rows with exactly 4 choices labelled
`A, B, C, D` in order, with `answerKey` one of those four letters.

In [5]:
from datasets import load_dataset

ds = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
print(f"raw test split : {len(ds)} rows")


def usable(row):
    return (len(row["choices"]["text"]) == 4
            and row["choices"]["label"] == LETTERS
            and row["answerKey"] in LETTERS)


items = [r for r in ds if usable(r)]
dropped = len(ds) - len(items)
print(f"usable (4 choices, labelled A-D) : {len(items)}   (dropped {dropped} malformed rows)")

items = items[:800]
print(f"using first {len(items)} usable questions")

assert len(items) >= 500, "Fewer than 500 usable questions - check the filter or fetch more rows"

Generating train split:   0%|          | 0/2251 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 2251/2251 [00:00<00:00, 59694.35 examples/s]

Generating test split:   0%|          | 0/2376 [00:00<?, ? examples/s]

Generating test split: 100%|██████████| 2376/2376 [00:00<00:00, 363869.81 examples/s]

Generating validation split:   0%|          | 0/570 [00:00<?, ? examples/s]

Generating validation split: 100%|██████████| 570/570 [00:00<00:00, 61557.06 examples/s]

raw test split : 2376 rows
usable (4 choices, labelled A-D) : 2268   (dropped 108 malformed rows)
using first 800 usable questions


## a2 sanity checks

Before scoring 800 questions, look at a few by eye and check the answer key isn't lopsided (e.g. almost
always "A") — a skewed key would make an unrelated bug look like a calibration finding later on.

In [6]:
import pandas as pd
from collections import Counter

print("first 3 questions:\n")
for r in items[:3]:
    body = "\n".join(f"  {L}. {t}" for L, t in zip(LETTERS, r["choices"]["text"]))
    print(f"[{r['id']}] {r['question']}\n{body}\n  answerKey: {r['answerKey']}\n")

key_counts = Counter(r["answerKey"] for r in items)
print("answer key distribution:", dict(sorted(key_counts.items())))
display(pd.Series(key_counts).sort_index().rename("count").to_frame())

first 3 questions:

[Mercury_417466] Which statement best explains why photosynthesis is the foundation of most food webs?
  A. Sunlight is the source of energy for nearly all ecosystems.
  B. Most ecosystems are found on land instead of in water.
  C. Carbon dioxide is more available than other gases.
  D. The producers in all ecosystems are plants.
  answerKey: A

[Mercury_7081673] Which piece of safety equipment is used to keep mold spores from entering the respiratory system?
  A. safety goggles
  B. breathing mask
  C. rubber gloves
  D. lead apron
  answerKey: B

[Mercury_7239733] Meiosis is a type of cell division in which germ cells divide to produce haploid cells. Where does meiosis occur?
  A. brain cells
  B. bone cells
  C. muscle cells
  D. ovary cells
  answerKey: D

answer key distribution: {'A': 191, 'B': 212, 'C': 207, 'D': 190}


,count
A,191
B,212
C,207
D,190


## What to check before moving on to a3

- `usable` count is comfortably within the 500-1000 target (protocol §2 / `CLAUDE.md` §2). If it's far
  outside that range, check the filter against the printed sample rows.
- The three printed questions read like well-formed 4-option science questions with a real `answerKey`.
- The answer-key distribution across A/B/C/D is roughly even. ARC is not adversarially balanced, so a
  moderate skew is normal, but a strong skew (e.g. one letter is 3x the others) is worth noting before
  building the option-order comparison (step c2), since it's exactly the kind of thing that comparison probes.
- `items` (a plain Python list of dataset rows) is what step a3's scoring function will iterate over.